# SQL Daily Review：每台设备最新有效记录

## 题目背景

设备在一天内可能产生多条巡检记录。

部分记录的状态为 `CANCELLED`，表示该次记录已经取消，不应参与最新状态判断。

现在需要从每台设备的有效记录中，找出最新的一条。

## 题目要求

先排除以下记录：

```text
status = 'CANCELLED'
```

然后找出每台设备最新的一条有效巡检记录。

### 排名规则

每台设备内部按照以下顺序判断记录的新旧：

1. 按照 `inspection_time` 降序排列；
2. 当 `inspection_time` 相同时，`record_id` 较大的记录优先。

每台设备最终只保留排名第 `1` 的一条记录。

### 输出字段

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号 |
| `record_id` | 记录编号 |
| `inspection_time` | 巡检时间 |
| `status` | 设备状态 |
| `visibility` | 能见度值 |

### 最终排序

最终结果按照 `device_id` 升序排列。

## 解题要求

- 使用窗口函数完成；
- 使用 `ROW_NUMBER()` 生成每台设备内部的记录排名；
- 不使用 `GROUP BY + MAX()` 后再关联原表的方法；
- 每台设备最终只能保留一条记录。

In [1]:
import pandas as pd
import duckdb

df_device_inspection = pd.DataFrame({
    "record_id": [
        101, 102, 103, 104,
        201, 202, 203,
        301, 302, 303
    ],
    "device_id": [
        "R34", "R34", "R34", "R34",
        "R16", "R16", "R16",
        "R05", "R05", "R05"
    ],
    "inspection_time": [
        "2026-07-20 08:00:00",
        "2026-07-20 12:00:00",
        "2026-07-20 12:00:00",
        "2026-07-20 16:00:00",
        "2026-07-20 09:00:00",
        "2026-07-20 14:00:00",
        "2026-07-20 18:00:00",
        "2026-07-20 07:30:00",
        "2026-07-20 15:30:00",
        "2026-07-20 15:30:00"
    ],
    "status": [
        "OK", "WARNING", "ERROR", "CANCELLED",
        "OK", "ERROR", "OK",
        "WARNING", "OK", "ERROR"
    ],
    "visibility": [
        1500, 800, 650, 700,
        2200, 900, 1800,
        1100, 2000, 750
    ]
})

df_device_inspection["inspection_time"] = pd.to_datetime(
    df_device_inspection["inspection_time"]
)

df_device_inspection

,record_id,device_id,inspection_time,status,visibility
0,101,R34,2026-07-20 08:00:00,OK,1500
1,102,R34,2026-07-20 12:00:00,WARNING,800
2,103,R34,2026-07-20 12:00:00,ERROR,650
3,104,R34,2026-07-20 16:00:00,CANCELLED,700
4,201,R16,2026-07-20 09:00:00,OK,2200
5,202,R16,2026-07-20 14:00:00,ERROR,900
6,203,R16,2026-07-20 18:00:00,OK,1800
7,301,R05,2026-07-20 07:30:00,WARNING,1100
8,302,R05,2026-07-20 15:30:00,OK,2000
9,303,R05,2026-07-20 15:30:00,ERROR,750


In [6]:
query = '''
WITH rank_table AS(
    SELECT
        device_id,
        record_id,
        inspection_time,
        status,
        visibility,
        ROW_NUMBER()
        OVER(PARTITION BY device_id ORDER BY inspection_time DESC,record_id DESC) AS time_rank
    FROM df_device_inspection
    WHERE status != 'CANCELLED'
)
SELECT 
    device_id,
        record_id,
        inspection_time,
        status,
        visibility
FROM rank_table
WHERE time_rank = 1
ORDER BY device_id

'''
df = duckdb.execute(query).fetchdf()
df

,device_id,record_id,inspection_time,status,visibility
0,R05,303,2026-07-20 15:30:00,ERROR,750
1,R16,203,2026-07-20 18:00:00,OK,1800
2,R34,103,2026-07-20 12:00:00,ERROR,650
